---
title: "2.4 Missing values and data types"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/02-data/04-missing-types.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/02-data/04-missing-types.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [2.3 Creating and transforming variables](../02-data/03-transforming.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Identify missing values and distinguish them from zero.
- Convert numeric text while counting failed conversions.
- Clean text categories and document coded missing responses.

## Why it matters

A spreadsheet can contain `"unknown"` in an age column, extra spaces in category labels, or -99 to mean “not reported.” Python cannot infer those meanings. A **missing value** means a measurement is absent or unknown, not necessarily zero. Start with made-up numbers, inspect a deliberately altered copy of the historical panel, then clean a clearly labelled simulated survey. No actual respondents or private data are used.

## Worked example

### Convert text deliberately

A **data type** describes the kind of values a column can hold. This fictional example stores ages as text, including one invalid answer. `pd.to_numeric` converts valid numeric text; `errors="coerce"` turns unconvertible values into missing values. Count conversions that failed rather than discarding them silently.

In [ ]:
import pandas as pd
small = pd.DataFrame({"respondent": ["Northland A", "Northland B", "Northland C"], "age_text": ["34", "unknown", "45"]})
small["age"] = pd.to_numeric(small["age_text"], errors="coerce")
print(small["age"].isna().sum())
small

One age is missing after conversion. `.isna()` returns a Boolean mask identifying missing values; `.sum()` counts True entries. Coercion makes a calculation possible, but it does not establish that an invalid answer should be ignored. Preserve the raw response for inspection.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Teach gaps without changing the real source

This filtered Gapminder teaching extract has no missing measurements in its six columns. To practise, deliberately blank one life-expectancy value in a separate four-row copy. `pd.NA` means a missing value. The original file is unchanged.

In [ ]:
countries = pd.read_csv(data_file)
print(countries.isna().sum())
practice_panel = countries.head(4).copy()
practice_panel.loc[practice_panel.index[0], "life_exp"] = pd.NA
practice_panel[["country", "year", "life_exp"]]

The original counts are all zero; the practice copy has one missing measurement. This is an invented gap in a real-data copy, not a new finding about Afghanistan. Never overwrite a raw measurement merely to make an exercise convenient.

### Prepare simulated person-level data

The next prepared cell locally generates 120 fictional records with a fixed random seed: the same starting state for the number generator, so you receive the same rows each time. It copies the course's committed generation function into the notebook to work without access to a private repository. Run it rather than memorizing it. Its numbers are simulated teaching data, not estimates of real wages or employment.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared simulated-data setup (run this cell)"
# Prepared setup: simulated people, generated locally; no real survey data.
from pathlib import Path
import csv
import io
import random

def generate_people():
    """Return a deterministic UTF-8 CSV using only Python's standard library."""
    rng = random.Random(20261111)
    output = io.StringIO(newline="")
    writer = csv.writer(output, lineterminator="\n")
    writer.writerow(["person_id", "age", "education", "education_years", "employed", "hourly_wage"])
    labels = {12: "secondary", 14: "college", 16: "university", 18: "postgraduate"}
    for person_id in range(1, 121):
        age = rng.randint(22, 64)
        education_years = rng.choice([12, 14, 16, 18])
        employed = int(rng.random() >= 0.2)
        wage = round(8 + 0.9 * education_years + 0.08 * age + rng.uniform(-2, 2), 2)
        # Blank wages for non-workers; -99 means a worker did not report a wage.
        wage_text = str(wage) if employed else ""
        if employed and person_id % 7 == 0:
            wage_text = "-99"
        education = labels[education_years]
        if person_id % 9 == 0:
            education = " " + education.upper() + " "
        age_text = str(age) if person_id % 13 else "unknown"
        writer.writerow([person_id, age_text, education, education_years, employed, wage_text])
    return output.getvalue().encode("utf-8")

people_file = Path("data/simulated_people.csv")
people_file.parent.mkdir(parents=True, exist_ok=True)
if not people_file.exists():
    people_file.write_bytes(generate_people())
print("Simulated data ready: 120 fictional people; not evidence about real wages.")

Look for **Simulated data ready**. [The simulation and codebook](https://github.com/ningzhanglab/Python-for-Beginners/blob/main/data/README.md) specify every deliberate problem. Reproducible fake data are useful for learning; a relationship built into their generator is not empirical evidence.

### Keep the raw response and a cleaned column

Read the simulated CSV. Its codebook says ages are whole years, the literal text `unknown` means an unknown age, and employed is coded 1 or 0. Keep the raw age column as text. Convert into a new column of **nullable integers**, whole numbers that can also represent missing values. The type label `Int64` has a capital I.

In [ ]:
people = pd.read_csv(people_file, dtype={"age": "string"})
people["age_clean"] = pd.to_numeric(people["age"], errors="coerce").astype("Int64")
print(people["age_clean"].isna().sum())
people[["person_id", "age", "age_clean"]].head(15)

There are nine unknown ages among the 120 simulated records. The raw text remains available for checking. An ordinary integer column cannot hold a missing value; nullable Int64 can. Numeric type alone does not confirm that an age is in a plausible range.

### Decode two kinds of absent wage

The simulated codebook defines a blank wage for a non-worker and -99 for a worker who did not report a wage. Neither should be treated as a real hourly wage. `.mask(condition)` replaces matching entries with missing values. Keep a separate flag for the special code so its reason is not lost.

In [ ]:
people["wage_not_reported"] = people["hourly_wage"] == -99
people["wage_clean"] = people["hourly_wage"].mask(people["wage_not_reported"])
print("Not-reported code:", people["wage_not_reported"].sum())
print("All missing clean wages:", people["wage_clean"].isna().sum())
print("Available wages:", people["wage_clean"].notna().sum())

The all-missing count includes both non-workers and workers with no reported wage. `.notna()` identifies available measurements. Do not fill these gaps with zero: that would mix non-employment, non-response and observed zero earnings. pandas' mean skips missing values, so any reported average must state which people contributed.

### Normalize and recode categories

A **category** is a named group. Clean formatting with `.str.strip()` (remove spaces at the ends) and `.str.lower()` (lowercase), then map known labels to an explicit code. `.map` applies a dictionary lookup to each value. Unexpected labels become missing after mapping and need review.

In [ ]:
people["education_clean"] = people["education"].str.strip().str.lower()
education_codes = {"secondary": 12, "college": 14, "university": 16, "postgraduate": 18}
people["education_years_checked"] = people["education_clean"].map(education_codes)
print(people["education_clean"].value_counts(dropna=False))
print("Unmapped categories:", people["education_years_checked"].isna().sum())

The cleaned labels collapse case/spacing variants into four documented groups, with no unmapped categories in this simulated file. The mapping uses this simulation's education-year convention; it is not an international education crosswalk. `dropna=False` keeps missing values visible in the frequency table.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Count available measurements

Count available `age_clean` values and compare with the total number of simulated records. Store the available count as `available_ages`.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
available_ages = people["age_clean"].notna().sum()
print("Available ages:", available_ages)
print("All simulated records:", len(people))

There are 111 available ages out of 120 simulated records. A denominator of 120 would be inappropriate for calculating a mean from only those 111 ages.

</details>

### 2. Make an analysis sample explicit

Create `wage_sample` containing simulated workers with an available cleaned wage. Print its row count and mean wage. Explain who is excluded.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
wage_sample = people.loc[(people["employed"] == 1) & people["wage_clean"].notna()].copy()
print(len(wage_sample))
print(round(wage_sample["wage_clean"].mean(), 2))

The mean describes simulated employed people who reported a wage. It excludes non-workers and worker non-respondents. It is not the mean of all 120 people, and the simulated dollar unit has no real price year or population interpretation.

</details>

### 3. Check a recode

Compare `education_years_checked` with the original simulated `education_years`. Print whether all records agree.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
education_agrees = (people["education_years_checked"] == people["education_years"]).all()
print(education_agrees)

The result is True. The redundant original field makes this teaching recode checkable. With a real survey, you would need the official codebook and a defensible conversion rather than this simulation-specific map.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### Numeric conversion is too strict for a dirty value

The float function cannot interpret a word as a number.

In [ ]:
try:
    float("unknown")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Inspect the raw entry and consult the codebook. For a column, pd.to_numeric with errors="coerce" is one explicit option, provided you count and review the resulting missing values.

### An unknown value is forced into a yes/no decision

The scalar pd.NA does not represent a known True or False.

In [ ]:
try:
    bool(pd.NA)
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

Missing means unknown. Test missingness with .isna() rather than asking whether pd.NA itself is True. Preserve unknown status when making indicators; do not automatically recode missing as zero.

## Recap

- Missing values and zero have different meanings.
- Preserve raw inputs, convert into new columns and count conversion failures.
- Decode special codes from a codebook and retain missingness reasons.
- Normalize category labels before an explicit, checked recode.

**Next:** [2.5 Group summaries: the pivot table lesson](../02-data/05-group-summaries.ipynb).

::: {.callout-note title="Coming from Stata / R / Excel?"}
Survey missing codes differ across tools and files. A Stata special missing value or an Excel blank is not automatically the same thing as a negative numeric code. Inspect the imported values and document their intended meanings.
:::